# Stokastisk 0.1.0 — Qwen2.5 inference on Kaggle 2× Tesla T4

This notebook installs the integrated Stokastisk/vLLM 0.25.1 wheel directly from the Hugging Face CDN, installs the release's 19 mandatory Python runtime dependencies, validates the Kaggle GPU environment, and runs Qwen2.5-0.5B-Instruct through the Stokastisk SDK with the **TRITON_ATTN** backend.

**Notebook design:** ordinary Kaggle cells only. No `main()` function, no `%%writefile`, no generated `.py` file, and no triple-quoted Python source passed to a subprocess.

**GPU choice:** Qwen2.5-0.5B-Instruct comfortably fits on one T4, so the recommended default is tensor parallel size 1. Both T4s are detected; the second remains available for another workload. An optional two-GPU configuration is shown near the end.

## 1. Confirm the Kaggle runtime

Enable **GPU** in the Kaggle notebook settings before running this cell. This release targets Python 3.13, CUDA 13.0, PyTorch 2.11.0+cu130, Triton 3.6.0, and Tesla T4 / SM75.

In [1]:
import sys
import torch
import triton

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('Triton:', triton.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Visible GPU count:', torch.cuda.device_count())

assert sys.version_info[:2] == (3, 13), 'This wheel targets Python 3.13.'
assert torch.__version__.startswith('2.11.0+cu130'), 'Unexpected PyTorch build; do not replace Kaggle CUDA packages blindly.'
assert torch.version.cuda == '13.0', 'This release targets CUDA 13.0.'
assert triton.__version__ == '3.6.0', 'This release targets Triton 3.6.0.'
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
assert torch.cuda.device_count() >= 2, 'This notebook expects Kaggle 2× Tesla T4 GPUs.'

for gpu_id in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(gpu_id)
    print(f'GPU {gpu_id}: {props.name}, compute capability {props.major}.{props.minor}, {props.total_memory / 2**30:.2f} GiB')
    assert props.name == 'Tesla T4', f'Expected Tesla T4, found {props.name}'
    assert (props.major, props.minor) == (7, 5), 'Expected T4 compute capability SM75.'

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA runtime: 13.0
Triton: 3.6.0
CUDA available: True
Visible GPU count: 2
GPU 0: Tesla T4, compute capability 7.5, 14.56 GiB
GPU 1: Tesla T4, compute capability 7.5, 14.56 GiB


## 2. Install the integrated Stokastisk 0.1.0 / vLLM 0.25.1 wheel from Github

In [2]:
!pip install stokastisk --extra-index-url https://ai-biruni.github.io/stokastisk/simple/

Looking in indexes: https://pypi.org/simple, https://ai-biruni.github.io/stokastisk/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 17.1 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 30.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 84.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 101.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 69.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 MB 139.0 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20/20 [stokastisk]0 [stokastisk]nforcer]]


## 3. Validate the installed SDK and the T4 backend policy

The integrated wheel contains the `stokastisk` SDK and the bundled vLLM 0.25.1 source/native components. This cell checks package versions, GPU visibility, and that the release's prohibited Flash-family backend report is clean.

In [3]:
import importlib.metadata as metadata
import json
import stokastisk
import torch
import triton

print('Stokastisk:', stokastisk.__version__)
print('Bundled vLLM distribution:', metadata.version('vllm'))
print('PyTorch:', torch.__version__)
print('Triton:', triton.__version__)
print('CUDA runtime:', torch.version.cuda)
print('GPU count:', torch.cuda.device_count())

assert stokastisk.__version__ == '0.1.0'
assert metadata.version('vllm') == '0.25.1'
assert torch.cuda.device_count() == 2

policy_report = stokastisk.prohibited_report()
print(json.dumps(policy_report, indent=2))
assert policy_report.get('clean') is True, 'A prohibited Flash-family component was detected.'

print('\nQwen2.5 model compatibility probe:')
print(json.dumps(stokastisk.check_model('Qwen/Qwen2.5-0.5B-Instruct'), indent=2, default=str))

Stokastisk: 0.1.0
Bundled vLLM distribution: 0.25.1
PyTorch: 2.11.0+cu130
Triton: 3.6.0
CUDA runtime: 13.0
GPU count: 2
{
  "distributions": [],
  "modules": [],
  "files": [],
  "native_hits": {},
  "binaries": [],
  "clean": true
}

Qwen2.5 model compatibility probe:
{
  "model": "Qwen/Qwen2.5-0.5B-Instruct",
  "dtype": "half",
  "quantization": {
    "method": null,
    "status": "SUPPORTED",
    "evidence": "kaggle",
    "note": "no quantization (all Kaggle runs)"
  },
  "architectures": [],
  "note": "architecture unknown: pass architectures=[...] or a local model directory (no network access is used)",
  "status": "UNKNOWN_REQUIRES_EVIDENCE"
}


## 4. Run Qwen2.5-0.5B-Instruct inference directly in this notebook cell

No subprocess wrapper or `main()` function is used. `stokastisk.LLM` applies the release defaults: FP16, eager execution, pinned `TRITON_ATTN`, and XGrammar structured outputs. The model weights are downloaded from Hugging Face on first use.

In [4]:
from stokastisk import LLM, StokastiskConfig

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'

# This model fits on one T4; TP=1 avoids unnecessary PCIe/NCCL overhead.
config = StokastiskConfig(
    tensor_parallel_size=1,
    dtype='half',
    attention_backend='TRITON_ATTN',
    max_model_len=2048,
    gpu_memory_utilization=0.88,
    enforce_eager=True,
    disable_custom_all_reduce=True,
)

with LLM(MODEL_ID, config=config) as llm:
    active_backend = llm.verify_attention_backend()
    print('Verified attention backend:', active_backend)
    assert active_backend == 'TRITON_ATTN'

    completion = llm.complete(
        'Explain in three short sentences why the sky appears blue.',
        max_tokens=96,
        temperature=0.2,
    )
    print('\n=== Completion ===\n', completion)

    chat_result = llm.chat([
        {'role': 'system', 'content': 'You are a concise assistant.'},
        {'role': 'user', 'content': 'Give me five practical uses of Python for data analysis.'},
    ])
    print('\n=== Chat ===\n', chat_result[0].outputs[0].text)

    schema = {
        'type': 'object',
        'properties': {
            'model': {'type': 'string', 'maxLength': 40},
            'gpu': {'type': 'string', 'maxLength': 40},
            'purpose': {'type': 'string', 'maxLength': 120},
        },
        'required': ['model', 'gpu', 'purpose'],
        'additionalProperties': False,
    }
    structured_result = llm.generate_structured(
        ['Return a short JSON object describing this Qwen2.5 inference test on a Tesla T4.'],
        schema,
        max_tokens=96,
        temperature=0.0,
    )
    print('\n=== Schema-constrained JSON ===\n', structured_result)

print('\nThe Stokastisk engine context has closed.')

INFO 11-14 13:24:41 [api_utils.py:273] non-default args: {'dtype': 'half', 'max_model_len': 2048, 'gpu_memory_utilization': 0.88, 'disable_log_stats': True, 'enforce_eager': True, 'disable_custom_all_reduce': True, 'structured_outputs_config': StructuredOutputsConfig(backend='xgrammar', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), 'attention_config': AttentionConfig(backend=<AttentionBackendEnum.TRITON_ATTN: 'vllm.v1.attention.backends.triton_attn.TritonAttentionBackend'>, flash_attn_version=None, use_prefill_decode_attention=False, flash_attn_max_num_splits_for_cuda_graph=32, tq_max_kv_splits_for_cuda_graph=32, use_trtllm_attention=None, disable_flashinfer_q_quantization=False, mla_prefill_backend=None, use_prefill_query_quantization=False, use_fp4_indexer_cache=False, indexer_kv_dtype='bf16', use_non_causal=False, flex_attn_block_m=None, flex_attn_block_n=None, flex_attn_q_block_size=No

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

INFO 11-14 13:24:58 [model.py:619] Resolved architecture: Qwen2ForCausalLM
WARNING 11-14 13:24:58 [model.py:2114] Casting torch.bfloat16 to torch.float16.
INFO 11-14 13:24:58 [model.py:1776] Using max model len 2048
INFO 11-14 13:24:58 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 11-14 13:24:58 [vllm.py:1042] Asynchronous scheduling is enabled.
WARNING 11-14 13:24:58 [vllm.py:1096] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 11-14 13:24:58 [vllm.py:1144] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 11-14 13:24:58 [kernel.py:292] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
INFO 11-14 13:24:58 [vllm.py:1322] Cudagraph is disabled under eager mode
INFO 11-14 13:

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (14 outputs left out)

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:06<00:00,  6.14s/it, est. speed input: 3.26 toks/s, output: 5.05 toks/s]


=== Schema-constrained JSON ===
 [{'model': 'Qwen2.5', 'gpu': 'False', 'purpose': 'Inference on a Tesla T4'}]
INFO 11-14 13:26:03 [core_client.py:655] [shutdown] MPClient: start timeout=default
INFO 11-14 13:26:03 [core_client.py:657] [shutdown] MPClient: stopping engine manager


(EngineCore pid=185) INFO 11-14 13:26:03 [core.py:1214] [shutdown] EngineCore: trigger received signal=SIGTERM
(EngineCore pid=185) INFO 11-14 13:26:03 [core.py:1333] [shutdown] EngineCore: start mode=abort timeout=0s
(EngineCore pid=185) INFO 11-14 13:26:03 [core.py:1364] [shutdown] EngineCore: request processing complete; starting resource teardown
(EngineCore pid=185) INFO 11-14 13:26:03 [core.py:1227] [shutdown] EngineCore: exiting busy loop
INFO 11-14 13:26:06 [core_client.py:659] [shutdown] MPClient: engine manager stopped
INFO 11-14 13:26:06 [core_client.py:660] [shutdown] MPClient: cleaning up background resources
INFO 11-14 13:26:06 [core_client.py:662] [shutdown] MPClient: complete

The Stokastisk engine context has closed.


## 5. Inspect GPU memory after inference


In [5]:
for gpu_id in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(gpu_id)
    free_bytes, total_bytes = torch.cuda.mem_get_info(gpu_id)
    print({
        'device': gpu_id,
        'name': props.name,
        'compute_capability': f'{props.major}.{props.minor}',
        'total_memory_GiB': round(total_bytes / 2**30, 2),
        'free_memory_GiB': round(free_bytes / 2**30, 2),
    })

{'device': 0, 'name': 'Tesla T4', 'compute_capability': '7.5', 'total_memory_GiB': 14.56, 'free_memory_GiB': 14.46}
{'device': 1, 'name': 'Tesla T4', 'compute_capability': '7.5', 'total_memory_GiB': 14.56, 'free_memory_GiB': 14.46}


## 6. Optional: distribute a larger model across both T4 GPUs

Do not run this cell for the 0.5B model unless you specifically want to test tensor parallelism. For a larger compatible model that needs both GPUs, change the config to `tensor_parallel_size=2`. Two T4s communicate over PCIe (not NVLink), so TP=2 can be slower than a single GPU when the model already fits on one card.

```python
config_2gpu = StokastiskConfig(
    tensor_parallel_size=2,
    dtype='half',
    attention_backend='TRITON_ATTN',
    max_model_len=2048,
    gpu_memory_utilization=0.88,
)
with LLM('Qwen/Qwen2.5-3B-Instruct', config=config_2gpu) as llm:
    print(llm.verify_attention_backend())
    print(llm.complete('Explain tensor parallelism briefly.', max_tokens=80))
```

The 3B example is illustrative, not a claim that every Qwen2.5 checkpoint or context length has been verified by this exact release.

In [6]:

# ============================================================
# STEP 6: TWO-GPU TENSOR-PARALLEL INFERENCE
# Stokastisk 0.1.0 | vLLM 0.25.1
# Python 3.13 | CUDA 13.0 | 2x NVIDIA Tesla T4 (SM75)
# ============================================================

import sys
import json
import torch
import stokastisk

from stokastisk import LLM, StokastiskConfig

# ------------------------------------------------------------
# 1. Runtime validation
# ------------------------------------------------------------

assert sys.version_info[:2] == (3, 13), "Python 3.13 required."
assert torch.cuda.is_available(), "CUDA is unavailable."
assert torch.cuda.device_count() == 2, "Exactly two visible GPUs required."
assert stokastisk.__version__ == "0.1.0"

print("Stokastisk:", stokastisk.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

for gpu_id in range(2):
    props = torch.cuda.get_device_properties(gpu_id)

    print(
        f"GPU {gpu_id}: {props.name} | "
        f"SM {props.major}.{props.minor} | "
        f"{props.total_memory / 2**30:.2f} GiB"
    )

    assert props.name == "Tesla T4"
    assert (props.major, props.minor) == (7, 5)

# ------------------------------------------------------------
# 2. Select a larger Qwen2.5 model
# ------------------------------------------------------------

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

compatibility = stokastisk.check_model(MODEL_ID)

print("\nModel compatibility:")
print(json.dumps(compatibility, indent=2, default=str))

# ------------------------------------------------------------
# 3. Configure tensor parallelism across both GPUs
# ------------------------------------------------------------

config_2gpu = StokastiskConfig(
    tensor_parallel_size=2,
    pipeline_parallel_size=1,

    dtype="half",

    attention_backend="TRITON_ATTN",

    max_model_len=2048,

    gpu_memory_utilization=0.88,

    enforce_eager=True,

    disable_custom_all_reduce=True,
)

print("\nTensor-parallel configuration:")
print(json.dumps(config_2gpu.as_dict(), indent=2, default=str))

# ------------------------------------------------------------
# 4. Launch the Stokastisk inference engine
# ------------------------------------------------------------

with LLM(MODEL_ID, config=config_2gpu) as llm:

    active_backend = llm.verify_attention_backend()

    print("\nVerified attention backend:", active_backend)

    assert active_backend == "TRITON_ATTN"

    # Completion
    completion = llm.complete(
        "Explain how tensor parallelism distributes "
        "a neural network across two GPUs.",
        max_tokens=128,
        temperature=0.2,
    )

    print("\n=== TWO-GPU COMPLETION ===")
    print(completion)

    # Chat
    chat_result = llm.chat([
        {
            "role": "system",
            "content": "You are a concise technical assistant."
        },
        {
            "role": "user",
            "content": "Explain the difference between "
                       "single-GPU inference and TP=2 inference."
        },
    ])

    print("\n=== TWO-GPU CHAT ===")
    print(chat_result[0].outputs[0].text)

    # --------------------------------------------------------
    # 5. Inspect GPU memory while the engine is active
    # --------------------------------------------------------

    print("\n=== GPU MEMORY DURING INFERENCE ===")

    for gpu_id in range(2):

        torch.cuda.synchronize(gpu_id)

        allocated = torch.cuda.memory_allocated(gpu_id)
        reserved = torch.cuda.memory_reserved(gpu_id)
        free, total = torch.cuda.mem_get_info(gpu_id)

        print({
            "gpu": gpu_id,
            "allocated_GiB": round(allocated / 2**30, 2),
            "reserved_GiB": round(reserved / 2**30, 2),
            "free_GiB": round(free / 2**30, 2),
            "total_GiB": round(total / 2**30, 2),
        })

print("\nTwo-GPU Stokastisk inference engine closed.")

# ------------------------------------------------------------
# 6. Confirm GPU memory after engine shutdown
# ------------------------------------------------------------

print("\n=== GPU MEMORY AFTER SHUTDOWN ===")

for gpu_id in range(2):

    free, total = torch.cuda.mem_get_info(gpu_id)

    print({
        "gpu": gpu_id,
        "free_GiB": round(free / 2**30, 2),
        "total_GiB": round(total / 2**30, 2),
    })

print("\nSTEP 6 COMPLETED.")


Stokastisk: 0.1.0
PyTorch: 2.11.0+cu130
CUDA: 13.0
GPU 0: Tesla T4 | SM 7.5 | 14.56 GiB
GPU 1: Tesla T4 | SM 7.5 | 14.56 GiB

Model compatibility:
{
  "model": "Qwen/Qwen2.5-3B-Instruct",
  "dtype": "half",
  "quantization": {
    "method": null,
    "status": "SUPPORTED",
    "evidence": "kaggle",
    "note": "no quantization (all Kaggle runs)"
  },
  "architectures": [],
  "note": "architecture unknown: pass architectures=[...] or a local model directory (no network access is used)",
  "status": "UNKNOWN_REQUIRES_EVIDENCE"
}

Tensor-parallel configuration:
{
  "tensor_parallel_size": 2,
  "pipeline_parallel_size": 1,
  "execution_strategy": null,
  "replica_devices": null,
  "dtype": "half",
  "kv_cache_dtype": "auto",
  "attention_backend": "TRITON_ATTN",
  "enforce_eager": true,
  "disable_custom_all_reduce": true,
  "gpu_memory_utilization": 0.88,
  "max_model_len": 2048,
  "structured_output_backend": "xgrammar",
  "apply_nccl_env": true,
  "nccl_env": {
    "NCCL_P2P_DISABLE": "

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

INFO 11-14 13:26:29 [model.py:619] Resolved architecture: Qwen2ForCausalLM
WARNING 11-14 13:26:29 [model.py:2114] Casting torch.bfloat16 to torch.float16.
INFO 11-14 13:26:29 [model.py:1776] Using max model len 2048
WARNING 11-14 13:26:29 [vllm.py:1096] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 11-14 13:26:29 [vllm.py:1144] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 11-14 13:26:29 [kernel.py:292] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
INFO 11-14 13:26:29 [vllm.py:1322] Cudagraph is disabled under eager mode


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (12 outputs left out)

Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.19s/it, est. speed input: 29.49 toks/s, output: 13.48 toks/s]


=== TWO-GPU CHAT ===
Single-GPU inference refers to the process of using a single GPU for executing an

=== GPU MEMORY DURING INFERENCE ===
{'gpu': 0, 'allocated_GiB': 0.0, 'reserved_GiB': 0.0, 'free_GiB': 1.09, 'total_GiB': 14.56}
{'gpu': 1, 'allocated_GiB': 0.0, 'reserved_GiB': 0.0, 'free_GiB': 1.09, 'total_GiB': 14.56}
INFO 11-14 13:28:22 [core_client.py:655] [shutdown] MPClient: start timeout=default


(EngineCore pid=458) INFO 11-14 13:28:22 [core.py:1214] [shutdown] EngineCore: trigger received signal=SIGTERM
(EngineCore pid=458) INFO 11-14 13:28:22 [core.py:1333] [shutdown] EngineCore: start mode=abort timeout=0s
(EngineCore pid=458) INFO 11-14 13:28:22 [core.py:1364] [shutdown] EngineCore: request processing complete; starting resource teardown
(EngineCore pid=458) INFO 11-14 13:28:22 [core.py:1227] [shutdown] EngineCore: exiting busy loop
(Worker_TP0 pid=483) INFO 11-14 13:28:22 [multiproc_executor.py:790] Parent process exited, terminating worker queues
(EngineCore pid=458) INFO 11-14 13:28:22 [multiproc_executor.py:426] [shutdown] Executor: waiting for worker exit count=2
(EngineCore pid=458) INFO 11-14 13:28:26 [multiproc_executor.py:433] [shutdown] Executor: all workers exited gracefully
WARNING 11-14 13:28:27 [utils.py:626] [shutdown] Process manager: force killing remaining processes count=1

Two-GPU Stokastisk inference engine closed.

=== GPU MEMORY AFTER SHUTDOWN ===
{'

## Release and installation notes

- Stokastisk repository: https://github.com/AI-Biruni/stokastisk
- Stokastisk v0.1.0 release: https://github.com/AI-Biruni/stokastisk/releases/tag/v0.1.0
- Hugging Face repository: https://huggingface.co/waqasm86/stokastisk
- Direct wheel CDN URL: https://huggingface.co/waqasm86/stokastisk/resolve/main/wheels/stokastisk-0.1.0-3-cp313-cp313-linux_x86_64.whl

**Important:** the previous attached notebook's inference cell completed Qwen2.5-0.5B-Instruct generation and logged `Using AttentionBackendEnum.TRITON_ATTN backend.` This new notebook removes the `main()` wrapper and `writefile` pattern as requested. Its direct-in-kernel inference cell is a new execution path and has not been run on Kaggle in this environment. Run the notebook from top to bottom in a fresh Kaggle session.


## 7. Pipeline parallelism across both NVIDIA Tesla T4 GPUs

This step tests pipeline parallelism (PP=2) using Stokastisk 0.1.0 and the bundled vLLM 0.25.1 runtime.

Unlike Step 6, which used tensor parallelism (TP=2), this step assigns sequential model layers to two pipeline stages:

- GPU 0: pipeline stage 0
- GPU 1: pipeline stage 1

### Configuration

- Model: `Qwen/Qwen2.5-3B-Instruct`
- Tensor parallel size: `1`
- Pipeline parallel size: `2`
- Precision: FP16 (`half`)
- Attention backend: `TRITON_ATTN`
- Maximum model length: `2048`
- GPU memory utilization: `0.88`
- Eager execution: enabled
- Custom all-reduce: disabled

The test uses the same model and inference configuration as Step 6, changing only the parallelism strategy. This makes the two tests easier to compare.

### Validation criteria

1. Python 3.13 and Stokastisk 0.1.0 are available.
2. Both GPUs are Tesla T4 devices with SM 7.5.
3. The configuration reports TP=1 and PP=2.
4. vLLM initializes with `tensor_parallel_size=1` and `pipeline_parallel_size=2`.
5. Both pipeline stages initialize successfully.
6. The attention backend is verified as `TRITON_ATTN`.
7. Completion and chat inference return generated text.
8. Both GPUs release their allocated memory after engine shutdown.

**Important:** This is a pipeline-parallel execution test, not a performance benchmark. Qwen2.5-3B-Instruct can fit on a single T4, so this model is used to compare the parallel execution modes rather than demonstrate the maximum model capacity of two GPUs.


In [7]:

# ============================================================
# STEP 7: PIPELINE PARALLELISM (PP=2)
# Stokastisk 0.1.0 | vLLM 0.25.1
# Python 3.13 | CUDA 13.0 | PyTorch 2.11.0
# Kaggle: 2x NVIDIA Tesla T4 (SM75)
# ============================================================

import json
import torch
import stokastisk

from stokastisk import LLM, StokastiskConfig


# ------------------------------------------------------------
# 1. Validate the Kaggle runtime
# ------------------------------------------------------------

print("=" * 65)
print("STEP 7: STOKASTISK PIPELINE PARALLELISM")
print("=" * 65)

print("Stokastisk:", stokastisk.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

assert stokastisk.__version__ == "0.1.0"
assert torch.cuda.is_available()
assert torch.cuda.device_count() == 2

assert torch.__version__.startswith("2.11.0")
assert torch.version.cuda == "13.0"

for gpu_id in range(2):

    props = torch.cuda.get_device_properties(gpu_id)

    print(
        f"GPU {gpu_id}: {props.name} | "
        f"SM {props.major}.{props.minor} | "
        f"{props.total_memory / 2**30:.2f} GiB"
    )

    assert props.name == "Tesla T4"
    assert (props.major, props.minor) == (7, 5)


# ------------------------------------------------------------
# 2. Select the same model used in Step 6
# ------------------------------------------------------------

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

compatibility = stokastisk.check_model(
    MODEL_ID,
    architectures=["Qwen2ForCausalLM"],
)

print("\nModel compatibility probe:")
print(json.dumps(compatibility, indent=2, default=str))


# ------------------------------------------------------------
# 3. Configure pipeline parallelism
# ------------------------------------------------------------

config_pp2 = StokastiskConfig(

    # One tensor-parallel partition.
    tensor_parallel_size=1,

    # Divide sequential model layers across two GPUs.
    pipeline_parallel_size=2,

    dtype="half",

    attention_backend="TRITON_ATTN",

    max_model_len=2048,

    gpu_memory_utilization=0.88,

    enforce_eager=True,

    disable_custom_all_reduce=True,
)


# ------------------------------------------------------------
# 4. Validate the configuration before engine initialization
# ------------------------------------------------------------

config_report = config_pp2.as_dict()

print("\nPipeline-parallel configuration:")
print(json.dumps(config_report, indent=2, default=str))

assert config_report["tensor_parallel_size"] == 1
assert config_report["pipeline_parallel_size"] == 2
assert config_report["attention_backend"] == "TRITON_ATTN"
assert config_report["dtype"] == "half"

print("\nTP=1 and PP=2 configuration validated.")


# ------------------------------------------------------------
# 5. Launch the pipeline-parallel inference engine
# ------------------------------------------------------------

with LLM(MODEL_ID, config=config_pp2) as llm:

    # Verify the attention backend selected by vLLM.
    active_backend = llm.verify_attention_backend()

    print("\nVerified attention backend:", active_backend)

    assert active_backend == "TRITON_ATTN"

    # --------------------------------------------------------
    # 5A. Completion inference
    # --------------------------------------------------------

    completion = llm.complete(

        "Explain pipeline parallelism in large language "
        "models. Describe how two NVIDIA GPUs process "
        "different sequential layers of the same model.",

        max_tokens=128,

        temperature=0.2,
    )

    print("\n" + "=" * 65)
    print("PIPELINE PARALLEL COMPLETION")
    print("=" * 65)

    print(completion)


    # --------------------------------------------------------
    # 5B. Chat inference
    # --------------------------------------------------------

    chat_result = llm.chat([

        {
            "role": "system",
            "content": "You are a concise technical assistant."
        },

        {
            "role": "user",
            "content": (
                "Compare tensor parallelism and pipeline "
                "parallelism when running an LLM on two GPUs."
            )
        },

    ])

    print("\n" + "=" * 65)
    print("PIPELINE PARALLEL CHAT")
    print("=" * 65)

    print(chat_result[0].outputs[0].text)


    # --------------------------------------------------------
    # 5C. Inspect GPU memory while the engine is active
    # --------------------------------------------------------

    print("\n" + "=" * 65)
    print("GPU MEMORY DURING PIPELINE INFERENCE")
    print("=" * 65)

    for gpu_id in range(2):

        torch.cuda.synchronize(gpu_id)

        free, total = torch.cuda.mem_get_info(gpu_id)

        allocated = torch.cuda.memory_allocated(gpu_id)
        reserved = torch.cuda.memory_reserved(gpu_id)

        print({

            "gpu": gpu_id,

            "name": torch.cuda.get_device_name(gpu_id),

            "allocated_GiB": round(
                allocated / 2**30, 2
            ),

            "reserved_GiB": round(
                reserved / 2**30, 2
            ),

            "free_GiB": round(
                free / 2**30, 2
            ),

            "total_GiB": round(
                total / 2**30, 2
            ),

        })


# ------------------------------------------------------------
# 6. Confirm engine shutdown and GPU memory recovery
# ------------------------------------------------------------

print("\nPipeline-parallel Stokastisk engine closed.")

print("\n" + "=" * 65)
print("GPU MEMORY AFTER ENGINE SHUTDOWN")
print("=" * 65)

for gpu_id in range(2):

    free, total = torch.cuda.mem_get_info(gpu_id)

    print({

        "gpu": gpu_id,

        "free_GiB": round(
            free / 2**30, 2
        ),

        "total_GiB": round(
            total / 2**30, 2
        ),

    })


# ------------------------------------------------------------
# 7. Final Step 7 result
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STEP 7 PIPELINE PARALLELISM TEST COMPLETED")
print("=" * 65)

print("Model:", MODEL_ID)
print("Tensor parallel size:", config_report["tensor_parallel_size"])
print("Pipeline parallel size:", config_report["pipeline_parallel_size"])
print("Attention backend:", active_backend)
print("GPU count:", torch.cuda.device_count())

print("\nExpected execution:")
print("GPU 0: Pipeline stage 0")
print("GPU 1: Pipeline stage 1")

print("\nSTEP 7 COMPLETED.")


STEP 7: STOKASTISK PIPELINE PARALLELISM
Stokastisk: 0.1.0
PyTorch: 2.11.0+cu130
CUDA: 13.0
GPU 0: Tesla T4 | SM 7.5 | 14.56 GiB
GPU 1: Tesla T4 | SM 7.5 | 14.56 GiB


pipeline_parallel is EXPERIMENTAL: Kaggle-verified on 2xT4 (notebooks 08, 13) but slower than TP=1 for a model that fits one T4; use it only for models that need two GPUs



Model compatibility probe:
{
  "model": "Qwen/Qwen2.5-3B-Instruct",
  "dtype": "half",
  "quantization": {
    "method": null,
    "status": "SUPPORTED",
    "evidence": "kaggle",
    "note": "no quantization (all Kaggle runs)"
  },
  "architectures": [
    {
      "architecture": "Qwen2ForCausalLM",
      "module": "qwen2",
      "status": "SUPPORTED",
      "evidence": "kaggle",
      "note": "Qwen2.5-0.5B/1.5B-Instruct generated on 2 x T4"
    }
  ],
  "status": "SUPPORTED"
}

Pipeline-parallel configuration:
{
  "tensor_parallel_size": 1,
  "pipeline_parallel_size": 2,
  "execution_strategy": null,
  "replica_devices": null,
  "dtype": "half",
  "kv_cache_dtype": "auto",
  "attention_backend": "TRITON_ATTN",
  "enforce_eager": true,
  "disable_custom_all_reduce": true,
  "gpu_memory_utilization": 0.88,
  "max_model_len": 2048,
  "structured_output_backend": "xgrammar",
  "apply_nccl_env": true,
  "nccl_env": {
    "NCCL_P2P_DISABLE": "1",
    "NCCL_IB_DISABLE": "1",
    "NCCL_CUME

(Worker_PP1 pid=1042) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
(Worker_PP0 pid=1041) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


(Worker_PP0 pid=1041) INFO 11-14 13:30:25 [weight_utils.py:849] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 5.75 GiB. Available RAM: 25.04 GiB.
(Worker_PP0 pid=1041) INFO 11-14 13:30:25 [weight_utils.py:872] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:04<00:04,  4.11s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:04<00:00,  2.07s/it]
(Worker_PP0 pid=1041) 


... (8 outputs left out)

Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.46it/s, est. speed input: 54.34 toks/s, output: 23.50 toks/s]


PIPELINE PARALLEL CHAT
Tensor parallelism and pipeline parallelism are techniques used to scale the training of large

GPU MEMORY DURING PIPELINE INFERENCE
{'gpu': 0, 'name': 'Tesla T4', 'allocated_GiB': 0.0, 'reserved_GiB': 0.0, 'free_GiB': 1.96, 'total_GiB': 14.56}
{'gpu': 1, 'name': 'Tesla T4', 'allocated_GiB': 0.0, 'reserved_GiB': 0.0, 'free_GiB': 1.48, 'total_GiB': 14.56}
INFO 11-14 13:30:46 [core_client.py:655] [shutdown] MPClient: start timeout=default


(EngineCore pid=1016) INFO 11-14 13:30:46 [core.py:1214] [shutdown] EngineCore: trigger received signal=SIGTERM
(EngineCore pid=1016) INFO 11-14 13:30:46 [core.py:1333] [shutdown] EngineCore: start mode=abort timeout=0s
(EngineCore pid=1016) INFO 11-14 13:30:46 [core.py:1364] [shutdown] EngineCore: request processing complete; starting resource teardown
(EngineCore pid=1016) INFO 11-14 13:30:46 [core.py:1227] [shutdown] EngineCore: exiting busy loop
(Worker_PP0 pid=1041) INFO 11-14 13:30:46 [multiproc_executor.py:790] Parent process exited, terminating worker queues
(EngineCore pid=1016) INFO 11-14 13:30:46 [multiproc_executor.py:426] [shutdown] Executor: waiting for worker exit count=2
(EngineCore pid=1016) INFO 11-14 13:30:50 [multiproc_executor.py:433] [shutdown] Executor: all workers exited gracefully
WARNING 11-14 13:30:51 [utils.py:626] [shutdown] Process manager: force killing remaining processes count=1

Pipeline-parallel Stokastisk engine closed.

GPU MEMORY AFTER ENGINE SHUTD